# APIs científicas con Python
## Práctica introductoria para estudiantes de nanociencias

Este cuaderno acompaña la presentación **APIs científicas con Python**. Incluye los ejemplos y una actividad con PubChem.

**Objetivo:** solicitar propiedades moleculares, interpretar una respuesta JSON y guardar los resultados como CSV.

**Cómo usarlo:** abre el archivo en Jupyter o Google Colab y ejecuta las celdas de arriba hacia abajo con `Shift + Enter`. Las celdas de texto explican el código. Las consultas a PubChem requieren internet.

Los archivos que guardemos aparecerán en la carpeta de trabajo del cuaderno. En Colab, descárgalos desde el panel **Archivos** antes de cerrar la sesión.

## 1. Preparación

`requests` permite hacer peticiones web. `pandas` organiza los datos en tablas.

Ejecuta la siguiente celda si necesitas instalar estas bibliotecas. `%pip` es una instrucción del cuaderno, no de un archivo `.py`.

In [1]:
%pip install requests pandas

In [2]:
import requests
import pandas as pd
import json
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

CARPETA = Path("resultados_apis")
CARPETA.mkdir(exist_ok=True)
print("Los resultados se guardarán en:", CARPETA.resolve())

Los resultados se guardarán en: /content/resultados_apis


## 2. Python mínimo

Una **variable** guarda un valor. El signo `=` asigna ese valor. `print()` lo muestra.

Un **diccionario** guarda pares clave y valor. Una **lista** guarda varios elementos. En Python, el primer elemento de una lista tiene índice `0`.

In [3]:
nombre = "ethanol"
print(nombre)

registro_ejemplo = {"formula": "C2H6O"}
print(registro_ejemplo["formula"])

nombres = ["ethanol", "citric acid"]
print(nombres[0])

ethanol
C2H6O
ethanol


## 3. JSON sin conectarnos a internet

JSON es un formato de intercambio de datos. `json.loads()` convierte un texto JSON en estructuras de Python.

Este registro es **un ejemplo didáctico**, no una descarga actual. La clave `CID` identifica un compuesto en PubChem. El valor de `MolecularWeight` llega aquí como texto.

In [4]:
texto_json = """{
  "CID": 702,
  "MolecularFormula": "C2H6O",
  "MolecularWeight": "46.07"
}"""

registro_ejemplo = json.loads(texto_json)
print(type(registro_ejemplo))
print(registro_ejemplo["MolecularFormula"])
print(float(registro_ejemplo["MolecularWeight"]))

<class 'dict'>
C2H6O
46.07


## 4. Primera petición a PubChem

En esta consulta, Python es el **cliente** y PubChem es el **servidor**.

- `GET` solicita información.
- La URL indica el compuesto, las propiedades y el formato de respuesta.
- `timeout=30` limita la espera de conexión o lectura. No es un límite total exacto de toda la descarga.
- `raise_for_status()` genera una excepción si el servidor responde con un error HTTP, como `404` o `503`.

Solicitaremos la fórmula y el peso molecular del etanol, sin clave de API.

In [5]:
base = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
ruta = "/compound/name/ethanol/property/"
campos = "MolecularFormula,MolecularWeight"
url = base + ruta + campos + "/JSON"

print("URL de la consulta:", url)
respuesta = requests.get(url, timeout=30)
print("Código de estado:", respuesta.status_code)
respuesta.raise_for_status()

URL de la consulta: https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/ethanol/property/MolecularFormula,MolecularWeight/JSON
Código de estado: 200


## 5. Lectura de la respuesta

`respuesta.json()` interpreta el contenido JSON y crea estructuras de Python. Primero observamos el contenido completo para conocer su organización.

In [6]:
datos = respuesta.json()
print(json.dumps(datos, indent=2, ensure_ascii=False))

{
  "PropertyTable": {
    "Properties": [
      {
        "CID": 702,
        "MolecularFormula": "C2H6O",
        "MolecularWeight": "46.07"
      }
    ]
  }
}


Las propiedades están dentro de `PropertyTable`, en la lista `Properties`. `[0]` selecciona el primer registro. Si una búsqueda devuelve varios compuestos, hay que verificar cuál corresponde a la identidad química deseada.

In [7]:
print("Claves principales:", list(datos.keys()))
registros = datos["PropertyTable"]["Properties"]
registro = registros[0]

print("CID:", registro["CID"])
print("Fórmula:", registro["MolecularFormula"])
print("Peso molecular:", registro["MolecularWeight"], "g/mol")

Claves principales: ['PropertyTable']
CID: 702
Fórmula: C2H6O
Peso molecular: 46.07 g/mol


## 6. Una tabla y un archivo CSV

Un `DataFrame` organiza los registros en filas y columnas. Convertimos explícitamente el peso molecular a número antes de hacer cálculos.

`index=False` evita guardar el índice interno de la tabla como una columna adicional. Las unidades de `MolecularWeight` son **g/mol** en esta práctica.

In [8]:
tabla = pd.DataFrame(registros)
tabla["MolecularWeight"] = pd.to_numeric(
    tabla["MolecularWeight"], errors="raise"
)

display(tabla)
archivo_etanol = CARPETA / "etanol.csv"
tabla.to_csv(archivo_etanol, index=False)
print("Archivo guardado:", archivo_etanol)

,CID,MolecularFormula,MolecularWeight
0,702,C2H6O,46.07


Archivo guardado: resultados_apis/etanol.csv


## 7. Consulta por lote

Podemos pedir varios compuestos en una sola solicitud. Usar identificadores CID evita repetir búsquedas por nombres ambiguos.

| CID | Compuesto |
|---:|---|
| 702 | Etanol |
| 311 | Ácido cítrico |
| 174 | Etilenglicol |

La API puede devolver los registros en un orden distinto al solicitado. Asociamos los nombres mediante su CID.

In [9]:
cids = "702,311,174"
campos = "MolecularFormula,MolecularWeight"
url_lote = (
    f"{base}/compound/cid/{cids}/property/"
    f"{campos}/JSON"
)

r = requests.get(url_lote, timeout=30)
r.raise_for_status()
fecha_lote = datetime.now(timezone.utc).isoformat()
datos_lote = r.json()
filas = datos_lote["PropertyTable"]["Properties"]
tabla_lote = pd.DataFrame(filas)
tabla_lote["MolecularWeight"] = pd.to_numeric(
    tabla_lote["MolecularWeight"], errors="raise"
)

nombres_por_cid = {
    702: "Etanol",
    311: "Ácido cítrico",
    174: "Etilenglicol",
}
tabla_lote["Nombre"] = tabla_lote["CID"].map(nombres_por_cid)
display(tabla_lote)

,CID,MolecularFormula,MolecularWeight,Nombre
0,702,C2H6O,46.07,Etanol
1,311,C6H8O7,192.12,Ácido cítrico
2,174,C2H6O2,62.07,Etilenglicol


## 8. Guardar el origen de los datos

Conservaremos la tabla procesada, la respuesta original y un registro de la consulta. Esto facilita repetir y revisar el trabajo.

El tiempo de consulta se registra en UTC. Si algún campo está ausente, hay que revisarlo antes de comparar compuestos.

In [ ]:
archivo_csv = CARPETA / "propiedades_lote.csv"
tabla_lote.to_csv(archivo_csv, index=False)

with (CARPETA / "respuesta_lote.json").open("w", encoding="utf-8") as archivo:
    json.dump(datos_lote, archivo, indent=2, ensure_ascii=False)

metadatos = {
    "fuente": "PubChem PUG REST",
    "url": url_lote,
    "fecha_consulta_utc": fecha_lote,
    "cids_solicitados": [702, 311, 174],
    "campos": ["MolecularFormula", "MolecularWeight"],
    "unidades": {"MolecularWeight": "g/mol"},
    "transformaciones": ["Conversión de MolecularWeight a número",
                          "Adición de nombres por CID"],
}
with (CARPETA / "consulta_lote.json").open("w", encoding="utf-8") as archivo:
    json.dump(metadatos, archivo, indent=2, ensure_ascii=False)

print("Archivos disponibles:")
for archivo in sorted(CARPETA.iterdir()):
    print(archivo.name)

## 9. Manejo básico de errores

`try` intenta ejecutar un bloque. `except` indica qué hacer si ocurre un error. La indentación agrupa las instrucciones de cada bloque.

Este ejemplo distingue espera excesiva, contenido JSON no válido y otros problemas de la petición. Si falla, el mensaje aparecerá debajo de la celda. **No hace reintentos automáticos.**

In [ ]:
try:
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    datos_error_ejemplo = r.json()
    print("Consulta correcta.")
except requests.exceptions.Timeout:
    print("El servicio tardó demasiado.")
except requests.exceptions.JSONDecodeError:
    print("La respuesta no contiene JSON válido.")
except requests.exceptions.RequestException as error:
    print("Falló la consulta:", error)

### Interpretación de errores frecuentes

| Código | Qué revisar |
|---|---|
| 400 | Formato, parámetros y propiedades solicitadas |
| 401 / 403 | Credenciales o permiso de acceso |
| 404 | Ruta e identidad del compuesto |
| 429 | Límite de solicitudes y tiempo de espera antes de reintentar |
| 5xx | Estado del servicio y posibilidad de intentarlo después |

Si una celda de consulta falla, corrige el problema y ejecútala de nuevo antes de continuar con las celdas que dependen de ella. `KeyError` indica que una clave esperada no existe en el contenido recibido.

PubChem pide no superar **5 solicitudes por segundo**. En una clase, varias computadoras pueden compartir la misma conexión de salida. Conviene usar lotes y conservar los resultados.

## 10. Actividad guiada

**Pregunta:** ¿qué fórmula y peso molecular tienen el ácido cítrico, el etilenglicol y el ácido oleico?

1. Verifica la identidad de los compuestos en PubChem y sus CID.
2. Completa las celdas de la actividad.
3. Obtén la tabla y guárdala como CSV.
4. Explica qué representa cada fila y qué unidades tiene el peso molecular.

| Compuesto | CID para verificar |
|---|---:|
| Ácido cítrico | 311 |
| Etilenglicol | 174 |
| Ácido oleico | 445639 |

En investigación, también importa verificar la forma química y el estado de protonación. Las propiedades moleculares no describen por sí solas el tamaño, la forma o la superficie de una nanopartícula.

In [ ]:
# Completa estos valores después de verificar los CID.
# Ejemplo de formato: [702, 311, 174]
cids_actividad = []

# La actividad se ejecutará cuando la lista tenga elementos.
if not cids_actividad:
    print("Completa cids_actividad y vuelve a ejecutar esta celda.")
else:
    cids_texto = ",".join(str(cid) for cid in cids_actividad)
    url_actividad = (
        f"{base}/compound/cid/{cids_texto}/property/"
        f"{campos}/JSON"
    )
    r_actividad = requests.get(url_actividad, timeout=30)
    r_actividad.raise_for_status()
    datos_actividad = r_actividad.json()
    tabla_actividad = pd.DataFrame(
        datos_actividad["PropertyTable"]["Properties"]
    )
    tabla_actividad["MolecularWeight"] = pd.to_numeric(
        tabla_actividad["MolecularWeight"], errors="raise"
    )
    display(tabla_actividad)
    tabla_actividad.to_csv(CARPETA / "actividad.csv", index=False)
    print("URL utilizada:", url_actividad)

### Preguntas para responder

- ¿Quién es el cliente y quién es el servidor?
- ¿Qué diferencia hay entre `GET` y `JSON`?
- ¿Por qué revisamos el estado antes de usar la respuesta?
- ¿Qué hace `[0]`?
- ¿Qué propiedades de la nanopartícula siguen faltando?
- ¿Cómo comprobarías que el compuesto descargado es el que buscas?

**Entrega:** cuaderno con tus respuestas, CSV de resultados y URL con fecha de consulta.

## 11. Solución de referencia de la actividad

Intenta la actividad antes de ejecutar esta celda. Reutilizamos los datos para crear una tabla con nombres. La condición evita hacer otra consulta si la actividad sigue vacía.

In [ ]:
# CID de referencia: 311 (ácido cítrico), 174 (etilenglicol),
# 445639 (ácido oleico).
if cids_actividad:
    nombres_actividad = {
        311: "Ácido cítrico",
        174: "Etilenglicol",
        445639: "Ácido oleico",
    }
    tabla_con_nombres = tabla_actividad.copy()
    tabla_con_nombres["Nombre"] = (
        tabla_con_nombres["CID"].map(nombres_actividad)
    )
    display(tabla_con_nombres)
else:
    print("Primero completa la actividad con [311, 174, 445639].")

## Documentación oficial

- [PubChem PUG REST](https://pubchem.ncbi.nlm.nih.gov/docs/pug-rest)
- [Tutorial de PubChem](https://pubchem.ncbi.nlm.nih.gov/docs/pug-rest-tutorial)
- [Requests: guía de uso](https://requests.readthedocs.io/en/latest/user/quickstart/)
- [pandas: introducción](https://pandas.pydata.org/docs/getting_started/index.html)
- [Materials Project: inicio con la API](https://docs.materialsproject.org/downloading-data/using-the-api/getting-started)

Este cuaderno utiliza PubChem para consultar moléculas. Materials Project ofrece datos de materiales mediante otra API y requiere su propia configuración y clave de acceso.